In [4]:
import json
import pickle
from pathlib import Path
from qdrant_client import QdrantClient
from qdrant_client.models import FieldCondition, Filter, MatchAny
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
COLLECTION_NAME = "southwind_docs"
RRF_K = 60

chunks = json.loads(Path("chunks.json").read_text(encoding="utf-8"))
chunk_by_id = {c["chunk_id"]: c for c in chunks}

with open("bm25_index.pkl", "rb") as f:
    bm25_bundle = pickle.load(f)

bm25: BM25Okapi = bm25_bundle["bm25"]
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")
qdrant = QdrantClient(path="qdrant_data")


def hybrid_search(query: str, role: str, top_k: int = 4, candidate_k: int = 8):
    role = role.lower().strip()

    # 1. Role-filtered dense search (Qdrant)
    q_vec = embed_model.encode(query, normalize_embeddings=True).tolist()
    dense_hits = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=q_vec,
        query_filter=Filter(
            must=[FieldCondition(key="allowed_roles", match=MatchAny(any=[role]))]
        ),
        limit=candidate_k,
    ).points
    dense_ranked = [h.payload["chunk_id"] for h in dense_hits]

    # 2. Role-filtered lexical search (BM25)
    q_tokens = tokenizer.tokenize(query.lower())
    bm25_scores = bm25.get_scores(q_tokens)
    bm25_sorted_indices = sorted(
        range(len(chunks)), key=lambda idx: bm25_scores[idx], reverse=True
    )
    bm25_ranked = [
        chunks[idx]["chunk_id"]
        for idx in bm25_sorted_indices
        if role in chunks[idx]["allowed_roles"]
    ][:candidate_k]

    # 3. Reciprocal Rank Fusion (RRF) over role-permitted candidates
    rrf_scores = {}
    for rank, cid in enumerate(dense_ranked, start=1):
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (RRF_K + rank)
    for rank, cid in enumerate(bm25_ranked, start=1):
        rrf_scores[cid] = rrf_scores.get(cid, 0.0) + 1.0 / (RRF_K + rank)

    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    return [
        {
            "chunk_id": cid,
            "rrf_score": round(score, 5),
            "roles": chunk_by_id[cid]["allowed_roles"],
            "section": chunk_by_id[cid]["section"],
            "text": chunk_by_id[cid]["text"],
        }
        for cid, score in fused
    ]


sample_query = "salary bank account wire transfer policy"
for r in ["employee", "manager", "admin"]:
    results = hybrid_search(sample_query, role=r, top_k=2)
    print(f"\n=== Role: {r} ===")
    for item in results:
        print(f"  {item['chunk_id']} | rrf={item['rrf_score']} | roles={item['roles']} | {item['section']}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1376.66it/s]



=== Role: employee ===
  sec3_chk0 | rrf=0.03279 | roles=['employee', 'manager', 'admin'] | [Page 4]
  sec1_chk0 | rrf=0.0315 | roles=['employee', 'manager', 'admin'] | [Page 2]

=== Role: manager ===
  sec2_chk0 | rrf=0.03252 | roles=['manager', 'admin'] | [Page 3]
  sec3_chk0 | rrf=0.03252 | roles=['employee', 'manager', 'admin'] | [Page 4]

=== Role: admin ===
  sec12_chk0 | rrf=0.03279 | roles=['admin'] | [Page 13]
  sec2_chk0 | rrf=0.032 | roles=['manager', 'admin'] | [Page 3]


In [5]:
# testing based on roles 
role_queries = {
    "employee": [
        "vendor third party risk customer data security",
        "phishing multi factor authentication incident root cause",
        "product roadmap engineering schedule updates",
    ],
    "manager": [
        "leave encashment casual sick leave policy",
        "performance review improvement plan team feedback",
        "backup disaster recovery postgresql schedule",
    ],
    "admin": [
        "finance approval spend limit lakh crore",
        "quarterly revenue outlook operating margin guidance",
        "on call rotation engineering runbook systems",
    ],
}

for role, queries in role_queries.items():
    print("=" * 72)
    print(f"ROLE: {role.upper()}")
    print("=" * 72)
    for q in queries:
        hits = hybrid_search(q, role=role, top_k=3, candidate_k=8)
        print(f"\nQuery: \"{q}\"")
        for idx, h in enumerate(hits, start=1):
            preview = " ".join(h["text"].split())[:110]
            print(
                f"  #{idx} | {h['chunk_id']:10s} | rrf={h['rrf_score']:.5f} | "
                f"roles={h['roles']} | {h['section']} | {preview}"
            )

ROLE: EMPLOYEE

Query: "vendor third party risk customer data security"
  #1 | sec4_chk0  | rrf=0.03279 | roles=['employee', 'manager', 'admin'] | [Page 5] | [ page 5 ] 5. device and network security all laptops use full - disk encryption and automatic screen lock aft
  #2 | sec3_chk0  | rrf=0.03200 | roles=['employee', 'manager', 'admin'] | [Page 4] | [ page 4 ] document 2 : information security policy access header : department security · allowed _ roles empl
  #3 | sec9_chk0  | rrf=0.03150 | roles=['employee', 'manager', 'admin'] | [Page 10] | [ page 10 ] 4. on - call rotation each service team keeps a weekly on - call rotation, changing every monday a

Query: "phishing multi factor authentication incident root cause"
  #1 | sec6_chk0  | rrf=0.03279 | roles=['employee', 'manager', 'admin'] | [Page 7] | [ page 7 ] root cause. the employee's account was protected by multi - factor authentication through text mess
  #2 | sec7_chk0  | rrf=0.03175 | roles=['employee', 'manager', 'admin']

In [3]:
from pathlib import Path
from pypdf import PdfReader

PDF_PATH = Path("Southwind Labs Company Documents.pdf")
reader = PdfReader(str(PDF_PATH))

for page_idx, page in enumerate(reader.pages, start=1):
    txt = page.extract_text() or ""
    lines = [ln.strip() for ln in txt.splitlines() if ln.strip()]
    matched = [
        ln for ln in lines
        if any(tok in ln.lower() for tok in ["document ", "access", "confidential", "employee", "manager", "admin"])
    ]
    print(f"--- Page {page_idx} ---")
    for m in matched[:6]:
        print("  ", m[:120])

--- Page 1 ---
   software company in Bengaluru with about 300 employees. Each document carries an
   access header (allowed roles and sensitivity) that becomes chunk metadata at ingestion.
   Document 1: HR Policy Handbook
   Access header: department hr · allowed_roles employee, manager, admin · sensitivity
   manager, admin · sensitivity restricted · contains_planted_secret true.
   Document ID: HR-001 · Owner: People Operations · Version 3.2 · Effective 1 April 2026
--- Page 2 ---
   department head as well as the line manager.
   discrimination, bullying and retaliation are not tolerated. Employees must declare
   Concerns can be raised with a manager, with HR, or anonymously through the ethics line
   assessment, a manager assessment and a calibration meeting across the department.
--- Page 3 ---
   or 2 triggers a 90-day improvement plan agreed between the employee, manager and HR.
   Employees may work remotely up to three days a week with manager approval. Fully
   employees r

In [7]:
#DEDUPLICATION AND POLISHING THE CHUNKS 
import hashlib
import json
import pickle
import re
from pathlib import Path
from qdrant_client.models import Distance, PointStruct, VectorParams
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer

EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
CHUNKS_JSON_PATH = Path("chunks.json")
BM25_PKL_PATH = Path("bm25_index.pkl")
COLLECTION_NAME = "southwind_docs"

raw_chunks = json.loads(CHUNKS_JSON_PATH.read_text(encoding="utf-8"))
tokenizer = AutoTokenizer.from_pretrained(EMBED_MODEL_NAME)


def normalize_text(text: str) -> str:
    # Collapse excessive spaces/newlines while keeping [Page X] markers readable
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def fingerprint(text: str) -> str:
    # Normalize case, punctuation spacing, and page tags to catch near-identical text
    cleaned = re.sub(r"\[page\s*\d+\]", " ", text.lower())
    cleaned = re.sub(r"[^a-z0-9]+", " ", cleaned).strip()
    return hashlib.sha256(cleaned.encode("utf-8")).hexdigest()


seen_hashes = set()
polished_chunks = []
dropped = []

for ch in raw_chunks:
    cleaned_text = normalize_text(ch["text"])
    fp = fingerprint(cleaned_text)

    if not cleaned_text or len(cleaned_text.split()) < 15:
        dropped.append((ch["chunk_id"], "too_short"))
        continue
    if fp in seen_hashes:
        dropped.append((ch["chunk_id"], "exact_or_normalized_duplicate"))
        continue

    seen_hashes.add(fp)
    updated = dict(ch)
    updated["text"] = cleaned_text
    updated["token_count"] = len(tokenizer.encode(cleaned_text, add_special_tokens=False))
    polished_chunks.append(updated)

print(f"Kept {len(polished_chunks)} / {len(raw_chunks)} chunks | Dropped: {dropped}")

CHUNKS_JSON_PATH.write_text(json.dumps(polished_chunks, indent=2, ensure_ascii=False), encoding="utf-8")

embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cpu")
embeddings = embed_model.encode(
    [c["text"] for c in polished_chunks],
    batch_size=8,
    normalize_embeddings=True,
    show_progress_bar=False,
)

if "client" in globals() and client is not None:
    if client.collection_exists(COLLECTION_NAME):
        client.delete_collection(COLLECTION_NAME)
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=embeddings.shape[1], distance=Distance.COSINE),
    )
    client.upsert(
        collection_name=COLLECTION_NAME,
        points=[
            PointStruct(
                id=i,
                vector=vec.tolist(),
                payload={
                    "chunk_id": c["chunk_id"],
                    "doc_id": c.get("doc_id"),
                    "section": c["section"],
                    "allowed_roles": c["allowed_roles"],
                    "token_count": c["token_count"],
                    "text": c["text"],
                },
            )
            for i, (c, vec) in enumerate(zip(polished_chunks, embeddings))
        ],
    )

tokenized_corpus = [tokenizer.tokenize(c["text"].lower()) for c in polished_chunks]
bm25 = BM25Okapi(tokenized_corpus)
with open(BM25_PKL_PATH, "wb") as f:
    pickle.dump({"bm25": bm25, "chunks": polished_chunks, "tokenized_corpus": tokenized_corpus}, f)

Kept 16 / 16 chunks | Dropped: []


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1219.62it/s]


In [8]:
#CROSS ENCODER 
import re
from sentence_transformers import CrossEncoder

CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device="cpu")


def deduplicate_hits(hits: list[dict]) -> list[dict]:
    seen = set()
    unique_hits = []
    for h in hits:
        norm = re.sub(r"\W+", " ", h["text"].lower()).strip()
        fp = hash(norm)
        if fp not in seen:
            seen.add(fp)
            unique_hits.append(h)
    return unique_hits


def rerank_and_select_pool(
    query: str,
    role: str,
    candidate_k: int = 8,
    top_pool: int = 4,
    min_ce_score: float = -5.0,
) -> list[dict]:
    # 1. Role-filtered hybrid search + RRF from your existing helper
    raw_hits = hybrid_search(query, role=role, top_k=candidate_k, candidate_k=candidate_k)

    # 2. Deduplicate candidate hits
    deduped = deduplicate_hits(raw_hits)
    if not deduped:
        return []

    # 3. Score (query, chunk_text) pairs with the cross-encoder
    pairs = [(query, h["text"]) for h in deduped]
    scores = cross_encoder.predict(pairs)

    rescored = []
    for h, ce_score in zip(deduped, scores):
        item = dict(h)
        item["ce_score"] = float(ce_score)
        rescored.append(item)

    rescored.sort(key=lambda x: x["ce_score"], reverse=True)

    # 4. Select high-quality pool (threshold + top_pool cap)
    hq_pool = [item for item in rescored if item["ce_score"] >= min_ce_score][:top_pool]

    # 5. Attach initial citation handles [S1], [S2], ... to high-quality pool chunks
    for idx, item in enumerate(hq_pool, start=1):
        doc_id = item.get("doc_id") or "DOC"
        section = item.get("section") or "[Page ?]"
        item["cite_id"] = f"S{idx}"
        item["citation_tag"] = f"[S{idx}: {item['chunk_id']} | {doc_id} | {section}]"

    return hq_pool


# Quick verification across roles
for r, q in [
    ("employee", "vendor third party risk customer data security"),
    ("manager", "performance review improvement plan team feedback"),
    ("admin", "finance approval spend limit lakh crore"),
]:
    pool = rerank_and_select_pool(q, role=r, candidate_k=8, top_pool=3)
    print(f"\nROLE={r} | QUERY=\"{q}\"")
    for p in pool:
        print(
            f"  {p['citation_tag']} | ce={p['ce_score']:.3f} | rrf={p['rrf_score']:.5f} | roles={p['roles']}"
        )

C:\Users\Lenovo\Desktop\hybrid-rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Lenovo\.cache\huggingface\hub\models--cross-encoder--ms-marco-MiniLM-L-6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\Lenovo\Desktop\hybrid-rag\.venv\Lib\site-packages\huggingface_hub\file_download.p


ROLE=employee | QUERY="vendor third party risk customer data security"
  [S1: sec3_chk0 | DOC | [Page 4]] | ce=-4.700 | rrf=0.03200 | roles=['employee', 'manager', 'admin']

ROLE=manager | QUERY="performance review improvement plan team feedback"
  [S1: sec2_chk0 | DOC | [Page 3]] | ce=-4.717 | rrf=0.03226 | roles=['manager', 'admin']

ROLE=admin | QUERY="finance approval spend limit lakh crore"
  [S1: sec12_chk0 | DOC | [Page 13]] | ce=5.572 | rrf=0.03279 | roles=['admin']
  [S2: sec13_chk0 | DOC | [Page 14]] | ce=-3.965 | rrf=0.03175 | roles=['employee', 'manager', 'admin']
  [S3: sec11_chk0 | DOC | [Page 12]] | ce=-4.367 | rrf=0.03126 | roles=['manager', 'admin']


In [9]:
# HIGH QULITY CHUNK SELECTION , CITATIONS , CONTEXT COMPRESSOR  TOKEN CONTROLLER
import json
import re
from pathlib import Path

CHUNKS_BY_ID = {
    c["chunk_id"]: c
    for c in json.loads(Path("chunks.json").read_text(encoding="utf-8"))
}

MAX_CONTEXT_TOKENS = 800
MAX_FRAMING_TOKENS = 300
MAX_GENERATION_TOKENS = 300
MAX_TOTAL_BUDGET = MAX_CONTEXT_TOKENS + MAX_FRAMING_TOKENS + MAX_GENERATION_TOKENS  # 1400 (< 1500)


def hydrate_metadata(item: dict) -> dict:
    enriched = dict(item)
    meta = CHUNKS_BY_ID.get(enriched["chunk_id"], {})
    if not enriched.get("doc_id") or enriched.get("doc_id") == "DOC":
        enriched["doc_id"] = meta.get("doc_id") or "UNKNOWN-DOC"
    if not enriched.get("section"):
        enriched["section"] = meta.get("section") or "[Page ?]"
    if not enriched.get("roles"):
        enriched["roles"] = meta.get("allowed_roles", [])
    return enriched


def select_high_quality_pool(
    reranked_hits: list[dict],
    top_pool: int = 4,
    min_ce_score: float = -10.0,
) -> list[dict]:
    """Select top-scoring chunks passing the cross-encoder cutoff and attach initial [S#] handles."""
    filtered = [hydrate_metadata(h) for h in reranked_hits if h.get("ce_score", -999.0) >= min_ce_score]
    filtered.sort(key=lambda x: x["ce_score"], reverse=True)
    hq_pool = filtered[:top_pool]

    for idx, item in enumerate(hq_pool, start=1):
        item["pool_rank"] = idx
        item["initial_cite_id"] = f"S{idx}"
        item["initial_citation_tag"] = f"[S{idx}: {item['chunk_id']} | {item['doc_id']} | {item['section']}]"

    return hq_pool


def compress_chunk_text(text: str, query: str, max_sentences: int = 5) -> str:
    """Light query-aware sentence compression while keeping [Page X] header and reading order."""
    lines = [ln.strip() for ln in text.splitlines() if ln.strip()]
    header = lines[0] if lines and re.match(r"^\[page\s*\d+\]$", lines[0].lower()) else ""
    body = " ".join(lines[1:] if header else lines)

    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", body) if s.strip()]
    if len(sents) <= max_sentences:
        compressed_body = body
    else:
        q_terms = {t for t in re.findall(r"[a-z0-9]+", query.lower()) if len(t) > 2}
        scored = []
        for idx, sent in enumerate(sents):
            s_terms = set(re.findall(r"[a-z0-9]+", sent.lower()))
            overlap = len(q_terms & s_terms)
            scored.append((overlap, -idx, sent))
        scored.sort(reverse=True)
        kept_indices = sorted(-item[1] for item in scored[:max_sentences])
        compressed_body = " ".join(sents[i] for i in kept_indices)

    return f"{header}\n{compressed_body}".strip() if header else compressed_body


def run_pool_compression_and_token_controller(
    query: str,
    role: str,
    candidate_k: int = 8,
    top_pool: int = 4,
    min_ce_score: float = -10.0,
    max_context_tokens: int = MAX_CONTEXT_TOKENS,
    max_framing_tokens: int = MAX_FRAMING_TOKENS,
    max_generation_tokens: int = MAX_GENERATION_TOKENS,
    compress: bool = True,
    max_sentences: int = 5,
) -> dict:
    # Step 1: Role-filtered hybrid RRF + deduplication + cross-encoder scoring
    raw_hits = hybrid_search(query, role=role, top_k=candidate_k, candidate_k=candidate_k)
    deduped = deduplicate_hits(raw_hits)

    rescored = []
    if deduped:
        pairs = [(query, h["text"]) for h in deduped]
        ce_scores = cross_encoder.predict(pairs)
        for h, score in zip(deduped, ce_scores):
            item = dict(h)
            item["ce_score"] = float(score)
            rescored.append(item)

    # Step 2: High-quality pool selection + initial [S#] citation assignment
    hq_pool = select_high_quality_pool(
        rescored,
        top_pool=top_pool,
        min_ce_score=min_ce_score,
    )

    # Step 3: Context compression + token controller (prunes citations for dropped chunks)
    packed_chunks = []
    pruned_chunks = []
    used_context_tokens = 0

    for item in hq_pool:
        compressed_text = (
            compress_chunk_text(item["text"], query=query, max_sentences=max_sentences)
            if compress
            else item["text"]
        )

        # Assign sequential final citation IDs [S1], [S2], ... strictly to packed chunks
        final_num = len(packed_chunks) + 1
        final_cite_id = f"S{final_num}"
        final_citation_tag = f"[{final_cite_id}: {item['chunk_id']} | {item['doc_id']} | {item['section']}]"
        formatted_block = f"{final_citation_tag}\n{compressed_text}"
        block_tokens = len(tokenizer.encode(formatted_block, add_special_tokens=False))

        if used_context_tokens + block_tokens <= max_context_tokens:
            entry = dict(item)
            entry["cite_id"] = final_cite_id
            entry["citation_tag"] = final_citation_tag
            entry["compressed_text"] = compressed_text
            entry["block_tokens"] = block_tokens
            packed_chunks.append(entry)
            used_context_tokens += block_tokens
        else:
            pruned_chunks.append(
                {
                    "dropped_initial_tag": item["initial_citation_tag"],
                    "chunk_id": item["chunk_id"],
                    "doc_id": item["doc_id"],
                    "ce_score": round(item["ce_score"], 3),
                    "reason": "exceeded_800_context_token_budget",
                }
            )

    final_context = "\n\n---\n\n".join(
        f"{c['citation_tag']}\n{c['compressed_text']}" for c in packed_chunks
    )
    active_citations = [c["citation_tag"] for c in packed_chunks]
    total_planned_tokens = used_context_tokens + max_framing_tokens + max_generation_tokens

    return {
        "role": role,
        "query": query,
        "hq_pool_count": len(hq_pool),
        "packed_count": len(packed_chunks),
        "used_context_tokens": used_context_tokens,
        "max_context_tokens": max_context_tokens,
        "max_framing_tokens": max_framing_tokens,
        "max_generation_tokens": max_generation_tokens,
        "total_planned_tokens": total_planned_tokens,
        "under_1500_cap": total_planned_tokens < 1500,
        "active_citations": active_citations,
        "pruned_chunks": pruned_chunks,
        "packed_chunks": packed_chunks,
        "context": final_context,
    }


# Test across roles
for r, q in [
    ("employee", "vendor third party risk customer data security"),
    ("manager", "performance review improvement plan team feedback"),
    ("admin", "finance approval spend limit lakh crore"),
]:
    out = run_pool_compression_and_token_controller(q, role=r)
    print("=" * 72)
    print(f"ROLE={r} | QUERY=\"{q}\"")
    print(
        f"Context tokens: {out['used_context_tokens']}/{out['max_context_tokens']} | "
        f"Planned total: {out['total_planned_tokens']} (<1500: {out['under_1500_cap']})"
    )
    print("Active citations:", out["active_citations"])
    print("Pruned chunks:", out["pruned_chunks"])
    print("\nFinalized Context Preview:\n", out["context"][:450], "\n")

ROLE=employee | QUERY="vendor third party risk customer data security"
Context tokens: 385/800 | Planned total: 985 (<1500: True)
Active citations: ['[S1: sec3_chk0 | SEC-001 | [Page 4]]', '[S2: sec13_chk0 | LDR-001 | [Page 14]]', '[S3: sec4_chk0 | SEC-001 | [Page 5]]']
Pruned chunks: []

Finalized Context Preview:
 [S1: sec3_chk0 | SEC-001 | [Page 4]]
[ page 4 ] document 2 : information security policy access header : department security · allowed _ roles employee, manager, admin · sensitivity internal. document id : sec - 001 · owner : security team · version 2. purpose and scope this policy protects the confidentiality, integrity and availability of southwind labs systems and customer data. data classification all company information belongs to one of fou 

ROLE=manager | QUERY="performance review improvement plan team feedback"
Context tokens: 210/800 | Planned total: 810 (<1500: True)
Active citations: ['[S1: sec2_chk0 | HR-001 | [Page 3]]', '[S2: sec1_chk0 | HR-001 | [Page 2]]']


In [12]:
import hashlib
import json
import os
import re
from urllib import error, request as urlrequest
from dotenv import load_dotenv

load_dotenv(dotenv_path=r"C:\Users\Lenovo\Desktop\hybrid-rag\.env", override=True)

assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found in .env — check C:\\Users\\Lenovo\\Desktop\\hybrid-rag\\.env"
print("Loaded .env successfully | Model:", os.getenv("GROQ_MODEL", "openai/gpt-oss-20b"))

ROLE_SCOPED_CACHE: dict[str, dict] = {}


def make_cache_key(role: str, query: str, packed_chunks: list[dict]) -> str:
    norm_q = " ".join(query.lower().split())
    chunk_sig = ",".join(sorted(c["chunk_id"] for c in packed_chunks))
    raw_key = f"role={role.strip().lower()}|q={norm_q}|chunks={chunk_sig}"
    return hashlib.sha256(raw_key.encode("utf-8")).hexdigest()


def sanitize_answer_citations(answer_text: str, valid_cite_ids: set[str]) -> str:
    """Keep only [S#] citations that match chunks present in the finalized context."""
    def _replace(match: re.Match) -> str:
        cite_id = match.group(1)
        return f"[{cite_id}]" if cite_id in valid_cite_ids else ""

    cleaned = re.sub(r"\[(S\d+)(?::[^\]]*)?\]", _replace, answer_text)
    cleaned = re.sub(r"[ \t]{2,}", " ", cleaned)
    cleaned = re.sub(r"\s+([.,;!?])", r"\1", cleaned)
    return cleaned.strip()


def call_groq_chat(
    system_prompt: str,
    user_prompt: str,
    max_tokens: int = MAX_GENERATION_TOKENS,
    model: str | None = None,
) -> str:
    api_key = os.getenv("GROQ_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError("GROQ_API_KEY is empty.")

    chosen_model = (model or os.getenv("GROQ_MODEL") or "openai/gpt-oss-20b").strip()
    payload = {
        "model": chosen_model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "max_tokens": max_tokens,
        "temperature": 0.2,
    }
    req = urlrequest.Request(
        "https://api.groq.com/openai/v1/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}",
            "User-Agent": "Southwind-Hybrid-RAG/1.0",
        },
        method="POST",
    )
    try:
        with urlrequest.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read().decode("utf-8"))
    except error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Groq API HTTP {exc.code}: {body}") from exc

    return data["choices"][0]["message"]["content"].strip()


def answer_with_rbac_rag(query: str, role: str, use_cache: bool = True) -> dict:
    ctx_bundle = run_pool_compression_and_token_controller(query=query, role=role)
    cache_key = make_cache_key(role, query, ctx_bundle["packed_chunks"])

    if use_cache and cache_key in ROLE_SCOPED_CACHE:
        cached = dict(ROLE_SCOPED_CACHE[cache_key])
        cached["cache_hit"] = True
        return cached

    valid_ids = {c["cite_id"] for c in ctx_bundle["packed_chunks"]}
    system_prompt = (
        f"You are a role-aware assistant for Southwind Labs answering for role='{role}'. "
        "Use ONLY the provided context blocks. Cite sources inline using [S1], [S2], etc. "
        "If the context does not contain the answer or is restricted for this role, state that clearly."
    )
    user_prompt = (
        f"Question: {query}\n\n"
        f"Authorized Context:\n{ctx_bundle['context'] if ctx_bundle['context'] else '(No authorized context)'}"
    )

    raw_answer = call_groq_chat(system_prompt, user_prompt, max_tokens=MAX_GENERATION_TOKENS)
    clean_answer = sanitize_answer_citations(raw_answer, valid_ids)

    result = {
        "role": role,
        "query": query,
        "cache_hit": False,
        "cache_key": cache_key[:12],
        "used_context_tokens": ctx_bundle["used_context_tokens"],
        "max_generation_tokens": MAX_GENERATION_TOKENS,
        "total_planned_tokens": ctx_bundle["total_planned_tokens"],
        "active_citations": ctx_bundle["active_citations"],
        "pruned_chunks": ctx_bundle["pruned_chunks"],
        "answer": clean_answer,
    }

    if use_cache:
        ROLE_SCOPED_CACHE[cache_key] = dict(result)

    return result


# Live verification on Groq: cache miss -> cache hit, plus RBAC across roles
test_cases = [
    ("admin", "What are the finance approval spend limits in lakh or crore?"),
    ("employee", "What are the finance approval spend limits in lakh or crore?"),
    ("employee", "What are third-party vendor risk and customer data security rules?"),
]

for role, q in test_cases:
    run1 = answer_with_rbac_rag(q, role=role)
    run2 = answer_with_rbac_rag(q, role=role)
    print("=" * 72)
    print(f"ROLE={role} | RUN1 cache_hit={run1['cache_hit']} | RUN2 cache_hit={run2['cache_hit']}")
    print(
        f"Context tokens: {run1['used_context_tokens']}/{MAX_CONTEXT_TOKENS} | "
        f"Planned total: {run1['total_planned_tokens']} (<1500: {run1['total_planned_tokens'] < 1500})"
    )
    print("Active citations:", run1["active_citations"])
    print("Answer:\n", run1["answer"], "\n")

Loaded .env successfully | Model: openai/gpt-oss-20b
ROLE=admin | RUN1 cache_hit=False | RUN2 cache_hit=True
Context tokens: 597/800 | Planned total: 1197 (<1500: True)
Active citations: ['[S1: sec12_chk0 | FIN-001 | [Page 13]]', '[S2: sec11_chk0 | FIN-001 | [Page 12]]', '[S3: sec13_chk0 | LDR-001 | [Page 14]]', '[S4: sec14_chk0 | LDR-001 | [Page 15]]']
Answer:
 The finance approval spend limits are:

- **₹ 10 lakh** – any single spend above this amount requires finance approval. 
- **₹ 50 lakh** – any single spend above this amount requires approval from both the CFO and the CEO. 

These thresholds are outlined in the finance policy document. [S1] 

ROLE=employee | RUN1 cache_hit=False | RUN2 cache_hit=True
Context tokens: 123/800 | Planned total: 723 (<1500: True)
Active citations: ['[S1: sec13_chk0 | LDR-001 | [Page 14]]']
Answer:
 I’m sorry, but the provided context does not contain any information about finance approval spend limits in lakh or crore. 

ROLE=employee | RUN1 cache_h

In [14]:
#adaptive compression and fitfull generation
def build_adaptive_context(
    query: str,
    role: str,
    candidate_k: int = 8,
    top_pool: int = 4,
    min_ce_score: float = -10.0,
    max_context_tokens: int = MAX_CONTEXT_TOKENS,
    max_framing_tokens: int = MAX_FRAMING_TOKENS,
    max_generation_tokens: int = MAX_GENERATION_TOKENS,
    max_sentences: int = 6,
) -> dict:
    raw_hits = hybrid_search(query, role=role, top_k=candidate_k, candidate_k=candidate_k)
    deduped = deduplicate_hits(raw_hits)

    rescored = []
    if deduped:
        pairs = [(query, h["text"]) for h in deduped]
        ce_scores = cross_encoder.predict(pairs)
        for h, score in zip(deduped, ce_scores):
            item = dict(h)
            item["ce_score"] = float(score)
            rescored.append(item)

    hq_pool = select_high_quality_pool(
        rescored,
        top_pool=top_pool,
        min_ce_score=min_ce_score,
    )

    def _pack(pool_items: list[dict], use_compression: bool) -> tuple[list[dict], list[dict], int]:
        packed, pruned, tokens_used = [], [], 0
        for item in pool_items:
            chunk_body = (
                compress_chunk_text(item["text"], query=query, max_sentences=max_sentences)
                if use_compression
                else item["text"]
            )
            cite_num = len(packed) + 1
            cite_id = f"S{cite_num}"
            citation_tag = f"[{cite_id}: {item['chunk_id']} | {item['doc_id']} | {item['section']}]"
            block = f"{citation_tag}\n{chunk_body}"
            block_tokens = len(tokenizer.encode(block, add_special_tokens=False))

            if tokens_used + block_tokens <= max_context_tokens:
                entry = dict(item)
                entry["cite_id"] = cite_id
                entry["citation_tag"] = citation_tag
                entry["final_text"] = chunk_body
                entry["block_tokens"] = block_tokens
                packed.append(entry)
                tokens_used += block_tokens
            else:
                pruned.append(
                    {
                        "dropped_initial_tag": item["initial_citation_tag"],
                        "chunk_id": item["chunk_id"],
                        "doc_id": item["doc_id"],
                        "ce_score": round(item["ce_score"], 3),
                        "reason": "exceeded_800_context_token_budget",
                    }
                )
        return packed, pruned, tokens_used

    # Pass 1: Try full (uncompressed) chunks first when they fit inside max_context_tokens
    packed_chunks, pruned_chunks, used_tokens = _pack(hq_pool, use_compression=False)
    compression_mode = "full_uncompressed"

    # Pass 2: If any pool chunk was pruned due to token budget, apply query-aware compression
    if pruned_chunks:
        comp_packed, comp_pruned, comp_tokens = _pack(hq_pool, use_compression=True)
        # Keep compressed pass if it packs more chunks or fits more evidence within budget
        if len(comp_packed) >= len(packed_chunks) and comp_tokens <= max_context_tokens:
            packed_chunks, pruned_chunks, used_tokens = comp_packed, comp_pruned, comp_tokens
            compression_mode = "query_aware_compressed"

    final_context = "\n\n---\n\n".join(
        f"{c['citation_tag']}\n{c['final_text']}" for c in packed_chunks
    )

    return {
        "role": role,
        "query": query,
        "compression_mode": compression_mode,
        "packed_count": len(packed_chunks),
        "used_context_tokens": used_tokens,
        "max_context_tokens": max_context_tokens,
        "max_framing_tokens": max_framing_tokens,
        "max_generation_tokens": max_generation_tokens,
        "total_planned_tokens": used_tokens + max_framing_tokens + max_generation_tokens,
        "active_citations": [c["citation_tag"] for c in packed_chunks],
        "pruned_chunks": pruned_chunks,
        "packed_chunks": packed_chunks,
        "context": final_context,
    }


def answer_with_rbac_rag_adaptive(query: str, role: str, use_cache: bool = True) -> dict:
    ctx_bundle = build_adaptive_context(query=query, role=role)
    cache_key = make_cache_key(
        f"{role}|mode={ctx_bundle['compression_mode']}",
        query,
        ctx_bundle["packed_chunks"],
    )

    if use_cache and cache_key in ROLE_SCOPED_CACHE:
        cached = dict(ROLE_SCOPED_CACHE[cache_key])
        cached["cache_hit"] = True
        return cached

    valid_ids = {c["cite_id"] for c in ctx_bundle["packed_chunks"]}
    system_prompt = (
        f"You are a role-aware assistant for Southwind Labs answering for role='{role}'. "
        "Use ONLY the provided context blocks. Provide a thorough, complete answer grounded in the context, "
        "and cite sources inline as [S1], [S2], etc. If restricted or absent from context, state that clearly."
    )
    user_prompt = (
        f"Question: {query}\n\n"
        f"Authorized Context:\n{ctx_bundle['context'] if ctx_bundle['context'] else '(No authorized context)'}"
    )

    raw_answer = call_groq_chat(system_prompt, user_prompt, max_tokens=MAX_GENERATION_TOKENS)
    clean_answer = sanitize_answer_citations(raw_answer, valid_ids)

    result = {
        "role": role,
        "query": query,
        "cache_hit": False,
        "compression_mode": ctx_bundle["compression_mode"],
        "used_context_tokens": ctx_bundle["used_context_tokens"],
        "max_generation_tokens": MAX_GENERATION_TOKENS,
        "total_planned_tokens": ctx_bundle["total_planned_tokens"],
        "active_citations": ctx_bundle["active_citations"],
        "pruned_chunks": ctx_bundle["pruned_chunks"],
        "answer": clean_answer,
    }

    if use_cache:
        ROLE_SCOPED_CACHE[cache_key] = dict(result)

    return result


# Re-run the employee vendor/security query + a heavy admin query to see both modes
for r, q in [
    ("employee", "What are third-party vendor risk and customer data security rules?"),
    ("admin", "Give a full overview of finance spend limits, vendor spend, leadership priorities, and security policies."),
]:
    res = answer_with_rbac_rag_adaptive(q, role=r)
    print("=" * 72)
    print(f"ROLE={r} | mode={res['compression_mode']} | tokens={res['used_context_tokens']}/{MAX_CONTEXT_TOKENS}")
    print(f"Planned total: {res['total_planned_tokens']} (<1500: {res['total_planned_tokens'] < 1500})")
    print("Active citations:", res["active_citations"], "| Pruned:", res["pruned_chunks"])
    print("Answer:\n", res["answer"], "\n")

ROLE=employee | mode=query_aware_compressed | tokens=504/800
Planned total: 1104 (<1500: True)
Active citations: ['[S1: sec3_chk0 | SEC-001 | [Page 4]]', '[S2: sec4_chk0 | SEC-001 | [Page 5]]', '[S3: sec13_chk0 | LDR-001 | [Page 14]]'] | Pruned: []
Answer:
 **Third‑party vendor risk rules** 
- The Information Security 

ROLE=admin | mode=query_aware_compressed | tokens=761/800
Planned total: 1361 (<1500: True)
Active citations: ['[S1: sec12_chk0 | FIN-001 | [Page 13]]', '[S2: sec14_chk0 | LDR-001 | [Page 15]]', '[S3: sec13_chk0 | LDR-001 | [Page 14]]', '[S4: sec3_chk0 | SEC-001 | [Page 4]]'] | Pruned: []
Answer:
  



In [15]:
import hashlib
import json
import os
import re
from urllib import error, request as urlrequest
from dotenv import load_dotenv

load_dotenv(dotenv_path=r"C:\Users\Lenovo\Desktop\hybrid-rag\.env", override=True)

assert os.getenv("GROQ_API_KEY"), "GROQ_API_KEY not found in .env — check C:\\Users\\Lenovo\\Desktop\\hybrid-rag\\.env"
print("Loaded .env successfully | Model:", os.getenv("GROQ_MODEL", "openai/gpt-oss-20b"))

ROLE_SCOPED_CACHE: dict[str, dict] = {}


def make_cache_key(role: str, query: str, packed_chunks: list[dict]) -> str:
    norm_q = " ".join(query.lower().split())
    chunk_sig = ",".join(sorted(c["chunk_id"] for c in packed_chunks))
    raw_key = f"role={role.strip().lower()}|q={norm_q}|chunks={chunk_sig}"
    return hashlib.sha256(raw_key.encode("utf-8")).hexdigest()


def sanitize_answer_citations(answer_text: str, valid_cite_ids: set[str]) -> str:
    """Keep only [S#] citations that match chunks present in the finalized context."""
    def _replace(match: re.Match) -> str:
        cite_id = match.group(1)
        return f"[{cite_id}]" if cite_id in valid_cite_ids else ""

    cleaned = re.sub(r"\[(S\d+)(?::[^\]]*)?\]", _replace, answer_text)
    cleaned = re.sub(r"[ \t]{2,}", " ", cleaned)
    cleaned = re.sub(r"\s+([.,;!?])", r"\1", cleaned)
    return cleaned.strip()


def call_groq_chat(
    system_prompt: str,
    user_prompt: str,
    max_tokens: int = MAX_GENERATION_TOKENS,
    model: str | None = None,
) -> str:
    api_key = os.getenv("GROQ_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError("GROQ_API_KEY is empty.")

    chosen_model = (model or os.getenv("GROQ_MODEL") or "openai/gpt-oss-20b").strip()
    payload = {
        "model": chosen_model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "max_tokens": max_tokens,
        "temperature": 0.2,
    }
    req = urlrequest.Request(
        "https://api.groq.com/openai/v1/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}",
            "User-Agent": "Southwind-Hybrid-RAG/1.0",
        },
        method="POST",
    )
    try:
        with urlrequest.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read().decode("utf-8"))
    except error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Groq API HTTP {exc.code}: {body}") from exc

    return data["choices"][0]["message"]["content"].strip()


def answer_with_rbac_rag(query: str, role: str, use_cache: bool = True) -> dict:
    ctx_bundle = run_pool_compression_and_token_controller(query=query, role=role)
    cache_key = make_cache_key(role, query, ctx_bundle["packed_chunks"])

    if use_cache and cache_key in ROLE_SCOPED_CACHE:
        cached = dict(ROLE_SCOPED_CACHE[cache_key])
        cached["cache_hit"] = True
        return cached

    valid_ids = {c["cite_id"] for c in ctx_bundle["packed_chunks"]}
    system_prompt = (
        f"You are a role-aware assistant for Southwind Labs answering for role='{role}'. "
        "Use ONLY the provided context blocks. Cite sources inline using [S1], [S2], etc. "
        "If the context does not contain the answer or is restricted for this role, state that clearly."
    )
    user_prompt = (
        f"Question: {query}\n\n"
        f"Authorized Context:\n{ctx_bundle['context'] if ctx_bundle['context'] else '(No authorized context)'}"
    )

    raw_answer = call_groq_chat(system_prompt, user_prompt, max_tokens=MAX_GENERATION_TOKENS)
    clean_answer = sanitize_answer_citations(raw_answer, valid_ids)

    result = {
        "role": role,
        "query": query,
        "cache_hit": False,
        "cache_key": cache_key[:12],
        "used_context_tokens": ctx_bundle["used_context_tokens"],
        "max_generation_tokens": MAX_GENERATION_TOKENS,
        "total_planned_tokens": ctx_bundle["total_planned_tokens"],
        "active_citations": ctx_bundle["active_citations"],
        "pruned_chunks": ctx_bundle["pruned_chunks"],
        "answer": clean_answer,
    }

    if use_cache:
        ROLE_SCOPED_CACHE[cache_key] = dict(result)

    return result


# Live verification on Groq: cache miss -> cache hit, plus RBAC across roles
test_cases = [
    ("admin", "What are the finance approval spend limits in lakh or crore?"),
    ("employee", "What are the finance approval spend limits in lakh or crore?"),
    ("employee", "What are third-party vendor risk and customer data security rules?"),
]

for role, q in test_cases:
    run1 = answer_with_rbac_rag(q, role=role)
    run2 = answer_with_rbac_rag(q, role=role)
    print("=" * 72)
    print(f"ROLE={role} | RUN1 cache_hit={run1['cache_hit']} | RUN2 cache_hit={run2['cache_hit']}")
    print(
        f"Context tokens: {run1['used_context_tokens']}/{MAX_CONTEXT_TOKENS} | "
        f"Planned total: {run1['total_planned_tokens']} (<1500: {run1['total_planned_tokens'] < 1500})"
    )
    print("Active citations:", run1["active_citations"])
    print("Answer:\n", run1["answer"], "\n")

Loaded .env successfully | Model: openai/gpt-oss-20b
ROLE=admin | RUN1 cache_hit=False | RUN2 cache_hit=True
Context tokens: 597/800 | Planned total: 1197 (<1500: True)
Active citations: ['[S1: sec12_chk0 | FIN-001 | [Page 13]]', '[S2: sec11_chk0 | FIN-001 | [Page 12]]', '[S3: sec13_chk0 | LDR-001 | [Page 14]]', '[S4: sec14_chk0 | LDR-001 | [Page 15]]']
Answer:
  

ROLE=employee | RUN1 cache_hit=False | RUN2 cache_hit=True
Context tokens: 123/800 | Planned total: 723 (<1500: True)
Active citations: ['[S1: sec13_chk0 | LDR-001 | [Page 14]]']
Answer:
 I’m sorry, but the provided context does not contain any information about finance approval spend limits. 

ROLE=employee | RUN1 cache_hit=False | RUN2 cache_hit=True
Context tokens: 425/800 | Planned total: 1025 (<1500: True)
Active citations: ['[S1: sec3_chk0 | SEC-001 | [Page 4]]', '[S2: sec4_chk0 | SEC-001 | [Page 5]]', '[S3: sec13_chk0 | LDR-001 | [Page 14]]']
Answer:
  



In [16]:
MAX_CONTEXT_TOKENS = 650
MAX_FRAMING_TOKENS = 250
MAX_GENERATION_TOKENS = 500


def call_groq_chat(
    system_prompt: str,
    user_prompt: str,
    max_tokens: int = MAX_GENERATION_TOKENS,
    model: str | None = None,
) -> str:
    api_key = os.getenv("GROQ_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError("GROQ_API_KEY is empty.")

    chosen_model = (model or os.getenv("GROQ_MODEL") or "openai/gpt-oss-20b").strip()
    payload = {
        "model": chosen_model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "max_tokens": max_tokens,
        "temperature": 0.2,
        "reasoning_effort": "low",
    }
    req = urlrequest.Request(
        "https://api.groq.com/openai/v1/chat/completions",
        data=json.dumps(payload).encode("utf-8"),
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {api_key}",
            "User-Agent": "Southwind-Hybrid-RAG/1.0",
        },
        method="POST",
    )
    try:
        with urlrequest.urlopen(req, timeout=30) as resp:
            data = json.loads(resp.read().decode("utf-8"))
    except error.HTTPError as exc:
        body = exc.read().decode("utf-8", errors="replace")
        # Fallback if reasoning_effort is rejected by the model/endpoint
        if exc.code == 400 and "reasoning_effort" in body:
            payload.pop("reasoning_effort", None)
            req2 = urlrequest.Request(
                "https://api.groq.com/openai/v1/chat/completions",
                data=json.dumps(payload).encode("utf-8"),
                headers={
                    "Content-Type": "application/json",
                    "Authorization": f"Bearer {api_key}",
                    "User-Agent": "Southwind-Hybrid-RAG/1.0",
                },
                method="POST",
            )
            with urlrequest.urlopen(req2, timeout=30) as resp2:
                data = json.loads(resp2.read().decode("utf-8"))
        else:
            raise RuntimeError(f"Groq API HTTP {exc.code}: {body}") from exc

    msg = data["choices"][0]["message"]
    return (msg.get("content") or "").strip()

In [18]:
def run_full_first_context(query: str, role: str, max_context_tokens: int = MAX_CONTEXT_TOKENS) -> dict:
    # Try uncompressed top_pool=2 first so full sentences stay intact when within budget
    bundle = build_adaptive_context(
        query=query,
        role=role,
        candidate_k=8,
        top_pool=2,
        min_ce_score=-10.0,
        max_context_tokens=max_context_tokens,
        max_framing_tokens=MAX_FRAMING_TOKENS,
        max_generation_tokens=MAX_GENERATION_TOKENS,
    )
    # If nothing was pruned, it ran cleanly in full_uncompressed mode
    if not bundle["pruned_chunks"]:
        return bundle

    # Otherwise widen pool to 4 with query-aware compression
    return build_adaptive_context(
        query=query,
        role=role,
        candidate_k=8,
        top_pool=4,
        min_ce_score=-10.0,
        max_context_tokens=max_context_tokens,
        max_framing_tokens=MAX_FRAMING_TOKENS,
        max_generation_tokens=MAX_GENERATION_TOKENS,
    )

In [19]:
#reasoning budget fix
MAX_CONTEXT_TOKENS = 650
MAX_FRAMING_TOKENS = 250
MAX_GENERATION_TOKENS = 500


def call_groq_chat(
    system_prompt: str,
    user_prompt: str,
    max_tokens: int = MAX_GENERATION_TOKENS,
    model: str | None = None,
) -> str:
    api_key = os.getenv("GROQ_API_KEY", "").strip()
    if not api_key:
        raise RuntimeError("GROQ_API_KEY is empty.")

    chosen_model = (model or os.getenv("GROQ_MODEL") or "openai/gpt-oss-20b").strip()
    payload = {
        "model": chosen_model,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        "max_tokens": max_tokens,
        "temperature": 0.2,
        "reasoning_effort": "low",
    }

    def _post(body_dict: dict) -> dict:
        req = urlrequest.Request(
            "https://api.groq.com/openai/v1/chat/completions",
            data=json.dumps(body_dict).encode("utf-8"),
            headers={
                "Content-Type": "application/json",
                "Authorization": f"Bearer {api_key}",
                "User-Agent": "Southwind-Hybrid-RAG/1.0",
            },
            method="POST",
        )
        with urlrequest.urlopen(req, timeout=30) as resp:
            return json.loads(resp.read().decode("utf-8"))

    try:
        data = _post(payload)
    except error.HTTPError as exc:
        err_body = exc.read().decode("utf-8", errors="replace")
        if exc.code == 400 and "reasoning_effort" in err_body:
            fallback_payload = dict(payload)
            fallback_payload.pop("reasoning_effort", None)
            data = _post(fallback_payload)
        else:
            raise RuntimeError(f"Groq API HTTP {exc.code}: {err_body}") from exc

    msg = data["choices"][0]["message"]
    return (msg.get("content") or "").strip()


# Quick verification on the two previously truncated queries
for r, q in [
    ("employee", "What are third-party vendor risk and customer data security rules?"),
    ("admin", "Give a full overview of finance spend limits, vendor spend, leadership priorities, and security policies."),
]:
    res = answer_with_rbac_rag_adaptive(q, role=r, use_cache=False)
    print("=" * 72)
    print(f"ROLE={r} | mode={res['compression_mode']} | tokens={res['used_context_tokens']}/{MAX_CONTEXT_TOKENS}")
    print(f"Planned total: {res['total_planned_tokens']} (<1500: {res['total_planned_tokens'] < 1500})")
    print("Active citations:", res["active_citations"])
    print("Answer:\n", res["answer"], "\n")

ROLE=employee | mode=query_aware_compressed | tokens=504/650
Planned total: 1104 (<1500: True)
Active citations: ['[S1: sec3_chk0 | SEC-001 | [Page 4]]', '[S2: sec4_chk0 | SEC-001 | [Page 5]]', '[S3: sec13_chk0 | LDR-001 | [Page 14]]']
Answer:
 **Third‑party vendor risk and customer data security rules**

Southwind Labs’ information‑security policy (SEC‑001) sets out the rules that apply to all employees, contractors and **vendors** who access company systems, as well as to all company‑owned and personal devices used for work. The policy is designed to protect the confidentiality, integrity and availability of Southwind Labs’ systems and **customer data**.

| Rule | What it means | Where it is documented |
|------|---------------|------------------------|
| **Scope** | The policy covers every person or entity that can access Southwind Labs’ systems – this includes third‑party vendors. | Sec‑001, page 4 – “purpose and scope” section [S1] |
| **Access control** | Access requests must be 

In [23]:
#MULTI SCENARIO EDGE CASE AND EVALUATION SUITE 
import time


def run_edge_case_evaluation_suite() -> None:
    ROLE_SCOPED_CACHE.clear()
    results_log = []

    def _record(case_id: str, title: str, passed: bool, detail: str) -> None:
        status = "PASS" if passed else "FAIL"
        results_log.append((case_id, title, status, detail))
        print(f"[{status}] {case_id} — {title}\n       {detail}")

    print("=" * 78)
    print("SOUTHWIND LABS RBAC HYBRID RAG — EDGE-CASE & EVALUATION SUITE")
    print("=" * 78)

    # 1. RBAC Strict Isolation: Employee blocked from Finance Spend Limits (FIN-001)
    r_emp_fin = answer_with_rbac_rag_adaptive(
        "What are the finance approval spend limits in lakh or crore?",
        role="employee",
    )
    leaked_fin = any("FIN-001" in c for c in r_emp_fin["active_citations"]) or ("50 lakh" in r_emp_fin["answer"].lower())
    _record(
        "EC-01",
        "RBAC Isolation (Employee -> Restricted FIN-001)",
        passed=(not leaked_fin and r_emp_fin["total_planned_tokens"] < 1500),
        detail=f"citations={r_emp_fin['active_citations']} | planned={r_emp_fin['total_planned_tokens']}",
    )

    # 2. RBAC Authorized Access: Admin retrieves FIN-001 + Project Lighthouse (LDR-001)
    r_adm_fin = answer_with_rbac_rag_adaptive(
        "What are the finance approval spend limits in lakh or crore?",
        role="admin",
    )
    has_fin = any("FIN-001" in c for c in r_adm_fin["active_citations"]) and ("10" in r_adm_fin["answer"])
    _record(
        "EC-02",
        "RBAC Authorized Access (Admin -> FIN-001)",
        passed=(has_fin and r_adm_fin["total_planned_tokens"] < 1500),
        detail=f"citations={r_adm_fin['active_citations']} | answer={r_adm_fin['answer'][:110]}...",
    )

    # 3. Mid-Tier Role Check: Manager Role Access Boundaries
    r_mgr = answer_with_rbac_rag_adaptive(
        "What are the rules for leave approval, hiring, and employee performance reviews?",
        role="manager",
    )
    mgr_ok = len(r_mgr["answer"]) > 20 and r_mgr["total_planned_tokens"] < 1500
    _record(
        "EC-03",
        "Manager Role Query & Token Budget",
        passed=mgr_ok,
        detail=f"mode={r_mgr['compression_mode']} | tokens={r_mgr['used_context_tokens']}/{MAX_CONTEXT_TOKENS} | citations={r_mgr['active_citations']}",
    )

    # 4. Out-of-Domain / No-Evidence Query (Hallucination & Refusal Guard)
    r_ood = answer_with_rbac_rag_adaptive(
        "What is Southwind Labs' cafeteria menu for Friday lunches in Tokyo?",
        role="admin",
    )
    refusal_markers = ("not contain", "no information", "not mentioned", "not provided", "not specified", "does not provide")
    refused_cleanly = any(m in r_ood["answer"].lower() for m in refusal_markers)
    _record(
        "EC-04",
        "Out-of-Domain Query Refusal (No Hallucination)",
        passed=refused_cleanly,
        detail=f"answer={r_ood['answer'][:130]}...",
    )

    # 5. Token Budget Stress & Citation Pruning on Broad Multi-Doc Query
    r_heavy = answer_with_rbac_rag_adaptive(
        "Summarize all rules across HR leave, security devices, AI tools, finance spend limits, and Project Lighthouse.",
        role="admin",
    )
    budget_ok = (
        r_heavy["used_context_tokens"] <= MAX_CONTEXT_TOKENS
        and r_heavy["total_planned_tokens"] < 1500
        and len(r_heavy["answer"].strip()) > 30
    )
    _record(
        "EC-05",
        "Token Budget Enforcement Under Heavy Multi-Doc Query",
        passed=budget_ok,
        detail=f"mode={r_heavy['compression_mode']} | context={r_heavy['used_context_tokens']}/{MAX_CONTEXT_TOKENS} | total={r_heavy['total_planned_tokens']} | pruned={len(r_heavy['pruned_chunks'])}",
    )

    # 6. Post-Generation Citation Sanitizer (Strips Ungrounded [S9] / [S99] Tags)
    fake_raw = "Employees must encrypt laptops [S1], report phishing [S2], and follow secret rule [S9] and [S99]."
    sanitized = sanitize_answer_citations(fake_raw, valid_cite_ids={"S1", "S2"})
    sanitizer_ok = ("[S1]" in sanitized) and ("[S2]" in sanitized) and ("[S9]" not in sanitized) and ("[S99]" not in sanitized)
    _record(
        "EC-06",
        "Post-Generation Citation Sanitization",
        passed=sanitizer_ok,
        detail=f"sanitized='{sanitized}'",
    )

    # 7. Role-Scoped Cache Isolation (Hit on Same Role, Miss on Different Role)
    r_cache_hit = answer_with_rbac_rag_adaptive(
        "What are the finance approval spend limits in lakh or crore?",
        role="admin",
    )
    r_cross_role = answer_with_rbac_rag_adaptive(
        "What are the finance approval spend limits in lakh or crore?",
        role="manager",
    )
    cache_ok = (r_cache_hit["cache_hit"] is True) and (r_cross_role["cache_hit"] is False)
    _record(
        "EC-07",
        "Role-Scoped Cache Isolation (Same-Role Hit vs. Cross-Role Miss)",
        passed=cache_ok,
        detail=f"admin_repeat_hit={r_cache_hit['cache_hit']} | manager_same_query_hit={r_cross_role['cache_hit']}",
    )

    passed_count = sum(1 for _, _, st, _ in results_log if st == "PASS")
    print("=" * 78)
    print(f"EVALUATION SUMMARY: {passed_count}/{len(results_log)} PASSED")
    print("=" * 78)


run_edge_case_evaluation_suite()

SOUTHWIND LABS RBAC HYBRID RAG — EDGE-CASE & EVALUATION SUITE
[PASS] EC-01 — RBAC Isolation (Employee -> Restricted FIN-001)
       citations=['[S1: sec13_chk0 | LDR-001 | [Page 14]]'] | planned=1147
[PASS] EC-02 — RBAC Authorized Access (Admin -> FIN-001)
       citations=['[S1: sec12_chk0 | FIN-001 | [Page 13]]', '[S2: sec11_chk0 | FIN-001 | [Page 12]]', '[S3: sec13_chk0 | LDR-001 | [Page 14]]', '[S4: sec14_chk0 | LDR-001 | [Page 15]]'] | answer=**Finance approval spend limits**

| Spend threshold | Approval required | Source |
|-----------------|-------...
[PASS] EC-03 — Manager Role Query & Token Budget
       mode=query_aware_compressed | tokens=647/650 | citations=['[S1: sec1_chk0 | HR-001 | [Page 2]]', '[S2: sec2_chk0 | HR-001 | [Page 3]]', '[S3: sec0_chk0 | HR-001 | [Page 1]]', '[S4: sec3_chk0 | SEC-001 | [Page 4]]']
[PASS] EC-04 — Out-of-Domain Query Refusal (No Hallucination)
       answer=I’m sorry, but the provided documents do not contain any information about Southwind La

In [22]:
def _fit_text_to_token_budget(text: str, query: str, token_budget: int) -> str:
    if token_budget <= 0:
        return ""
    if len(tokenizer.encode(text, add_special_tokens=False)) <= token_budget:
        return text
    # Progressively tighten sentence count until it fits token_budget
    for n_sent in (5, 4, 3, 2, 1):
        cand = compress_chunk_text(text, query=query, max_sentences=n_sent)
        if len(tokenizer.encode(cand, add_special_tokens=False)) <= token_budget:
            return cand
    # Hard token fallback if a single sentence is still too long
    token_ids = tokenizer.encode(text, add_special_tokens=False)[:token_budget]
    return tokenizer.decode(token_ids, skip_special_tokens=True).strip()


def build_adaptive_context(
    query: str,
    role: str,
    candidate_k: int = 8,
    top_pool: int = 4,
    min_ce_score: float = -10.0,
    max_context_tokens: int = MAX_CONTEXT_TOKENS,
    max_framing_tokens: int = MAX_FRAMING_TOKENS,
    max_generation_tokens: int = MAX_GENERATION_TOKENS,
    max_sentences: int = 6,
) -> dict:
    raw_hits = hybrid_search(query, role=role, top_k=candidate_k, candidate_k=candidate_k)
    deduped = deduplicate_hits(raw_hits)

    rescored = []
    if deduped:
        pairs = [(query, h["text"]) for h in deduped]
        ce_scores = cross_encoder.predict(pairs)
        for h, score in zip(deduped, ce_scores):
            item = dict(h)
            item["ce_score"] = float(score)
            rescored.append(item)

    hq_pool = select_high_quality_pool(
        rescored,
        top_pool=top_pool,
        min_ce_score=min_ce_score,
    )

    def _pack(pool_items: list[dict], use_compression: bool) -> tuple[list[dict], list[dict], int]:
        packed, pruned, tokens_used = [], [], 0
        for item in pool_items:
            cite_num = len(packed) + 1
            cite_id = f"S{cite_num}"
            citation_tag = f"[{cite_id}: {item['chunk_id']} | {item['doc_id']} | {item['section']}]"
            header_tokens = len(tokenizer.encode(f"{citation_tag}\n", add_special_tokens=False))
            remaining_budget = max_context_tokens - tokens_used - header_tokens

            if remaining_budget <= 15:
                pruned.append({
                    "dropped_initial_tag": item["initial_citation_tag"],
                    "chunk_id": item["chunk_id"],
                    "doc_id": item["doc_id"],
                    "ce_score": round(item["ce_score"], 3),
                    "reason": "context_token_budget_exhausted",
                })
                continue

            if use_compression:
                raw_body = compress_chunk_text(item["text"], query=query, max_sentences=max_sentences)
                chunk_body = _fit_text_to_token_budget(raw_body, query=query, token_budget=remaining_budget)
            else:
                chunk_body = item["text"]

            block = f"{citation_tag}\n{chunk_body}"
            block_tokens = len(tokenizer.encode(block, add_special_tokens=False))

            if tokens_used + block_tokens <= max_context_tokens:
                entry = dict(item)
                entry["cite_id"] = cite_id
                entry["citation_tag"] = citation_tag
                entry["final_text"] = chunk_body
                entry["block_tokens"] = block_tokens
                packed.append(entry)
                tokens_used += block_tokens
            else:
                pruned.append({
                    "dropped_initial_tag": item["initial_citation_tag"],
                    "chunk_id": item["chunk_id"],
                    "doc_id": item["doc_id"],
                    "ce_score": round(item["ce_score"], 3),
                    "reason": "exceeded_context_token_budget",
                })
        return packed, pruned, tokens_used

    packed_chunks, pruned_chunks, used_tokens = _pack(hq_pool, use_compression=False)
    compression_mode = "full_uncompressed"

    if pruned_chunks:
        comp_packed, comp_pruned, comp_tokens = _pack(hq_pool, use_compression=True)
        if comp_tokens <= max_context_tokens and (
            len(comp_packed) > len(packed_chunks)
            or (len(comp_packed) == len(packed_chunks) and comp_tokens >= used_tokens)
        ):
            packed_chunks, pruned_chunks, used_tokens = comp_packed, comp_pruned, comp_tokens
            compression_mode = "query_aware_compressed"

    assert used_tokens <= max_context_tokens, f"Context budget breached: {used_tokens} > {max_context_tokens}"

    final_context = "\n\n---\n\n".join(
        f"{c['citation_tag']}\n{c['final_text']}" for c in packed_chunks
    )

    return {
        "role": role,
        "query": query,
        "compression_mode": compression_mode,
        "packed_count": len(packed_chunks),
        "used_context_tokens": used_tokens,
        "max_context_tokens": max_context_tokens,
        "max_framing_tokens": max_framing_tokens,
        "max_generation_tokens": max_generation_tokens,
        "total_planned_tokens": used_tokens + max_framing_tokens + max_generation_tokens,
        "active_citations": [c["citation_tag"] for c in packed_chunks],
        "pruned_chunks": pruned_chunks,
        "packed_chunks": packed_chunks,
        "context": final_context,
    }

In [24]:
#END to END latency portfolio 
import time


def answer_with_telemetry(
    query: str,
    role: str,
    use_cache: bool = True,
) -> dict:
    t_start = time.perf_counter()

    # Stage A: Retrieval + Deduplication + Reranking
    t0 = time.perf_counter()
    raw_hits = hybrid_search(query, role=role, top_k=8, candidate_k=8)
    deduped = deduplicate_hits(raw_hits)
    rescored = []
    if deduped:
        pairs = [(query, h["text"]) for h in deduped]
        ce_scores = cross_encoder.predict(pairs)
        for h, score in zip(deduped, ce_scores):
            item = dict(h)
            item["ce_score"] = float(score)
            rescored.append(item)
    t_retrieval_rerank = (time.perf_counter() - t0) * 1000.0

    # Stage B: Context Packing & Adaptive Compression
    t1 = time.perf_counter()
    hq_pool = select_high_quality_pool(rescored, top_pool=4, min_ce_score=-10.0)
    # Reuse your existing build_adaptive_context logic or pack directly
    ctx_bundle = build_adaptive_context(query=query, role=role)
    t_pack = (time.perf_counter() - t1) * 1000.0

    cache_key = make_cache_key(
        f"{role}|mode={ctx_bundle['compression_mode']}",
        query,
        ctx_bundle["packed_chunks"],
    )

    if use_cache and cache_key in ROLE_SCOPED_CACHE:
        cached = dict(ROLE_SCOPED_CACHE[cache_key])
        total_ms = round((time.perf_counter() - t_start) * 1000.0, 2)
        cached["cache_hit"] = True
        cached["telemetry"] = {
            "retrieval_rerank_ms": round(t_retrieval_rerank, 2),
            "context_pack_ms": round(t_pack, 2),
            "llm_ms": 0.0,
            "total_ms": total_ms,
            "actual_prompt_tokens": None,
            "actual_completion_tokens": None,
            "actual_total_tokens": None,
        }
        return cached

    # Stage C: Live LLM Generation + Citation Sanitization
    t2 = time.perf_counter()
    valid_ids = {c["cite_id"] for c in ctx_bundle["packed_chunks"]}
    system_prompt = (
        f"You are a role-aware assistant for Southwind Labs answering for role='{role}'. "
        "Use ONLY the provided context blocks. Provide a clear, complete answer grounded in the context, "
        "and cite sources inline strictly in brackets like [S1], [S2]. If restricted or absent from context, state that clearly."
    )
    user_prompt = (
        f"Question: {query}\n\n"
        f"Authorized Context:\n{ctx_bundle['context'] if ctx_bundle['context'] else '(No authorized context)'}"
    )

    # Call Groq chat and capture usage if your call_groq_chat returns a string
    raw_answer = call_groq_chat(system_prompt, user_prompt, max_tokens=MAX_GENERATION_TOKENS)
    clean_answer = sanitize_answer_citations(raw_answer, valid_ids)
    t_llm = (time.perf_counter() - t2) * 1000.0
    total_ms = (time.perf_counter() - t_start) * 1000.0

    result = {
        "role": role,
        "query": query,
        "cache_hit": False,
        "compression_mode": ctx_bundle["compression_mode"],
        "used_context_tokens": ctx_bundle["used_context_tokens"],
        "max_generation_tokens": MAX_GENERATION_TOKENS,
        "total_planned_tokens": ctx_bundle["total_planned_tokens"],
        "active_citations": ctx_bundle["active_citations"],
        "pruned_chunks": ctx_bundle["pruned_chunks"],
        "answer": clean_answer,
        "telemetry": {
            "retrieval_rerank_ms": round(t_retrieval_rerank, 2),
            "context_pack_ms": round(t_pack, 2),
            "llm_ms": round(t_llm, 2),
            "total_ms": round(total_ms, 2),
        },
    }

    if use_cache:
        ROLE_SCOPED_CACHE[cache_key] = dict(result)

    return result


# Benchmark across roles + cache hit
ROLE_SCOPED_CACHE.clear()
benchmark_queries = [
    ("employee", "What are the rules for leave approval and laptop security?", True),
    ("manager", "What are the rules for leave approval, hiring, and employee performance reviews?", True),
    ("admin", "What are the finance approval spend limits in lakh or crore?", True),
    ("admin", "What are the finance approval spend limits in lakh or crore?", True),  # Cache hit check
]

print(f"{'Role':<9} | {'Cache':<5} | {'Mode':<22} | {'Ret+RR(ms)':<10} | {'LLM(ms)':<9} | {'Total(ms)':<9} | {'Ctx/Plan'}")
print("-" * 98)
for role, q, c_flag in benchmark_queries:
    out = answer_with_telemetry(q, role=role, use_cache=c_flag)
    tel = out["telemetry"]
    print(
        f"{role:<9} | {str(out['cache_hit']):<5} | {out['compression_mode']:<22} | "
        f"{tel['retrieval_rerank_ms']:<10} | {tel['llm_ms']:<9} | {tel['total_ms']:<9} | "
        f"{out['used_context_tokens']}/{out['total_planned_tokens']}"
    )

Role      | Cache | Mode                   | Ret+RR(ms) | LLM(ms)   | Total(ms) | Ctx/Plan
--------------------------------------------------------------------------------------------------
employee  | False | query_aware_compressed | 1926.93    | 3551.56   | 6794.44   | 644/1394
manager   | False | query_aware_compressed | 1327.8     | 1478.2    | 4047.55   | 647/1397
admin     | False | query_aware_compressed | 1257.91    | 1606.55   | 4127.49   | 643/1393
admin     | True  | query_aware_compressed | 1211.39    | 0.0       | 2423.75   | 643/1393
